# Context-Fabric MCP + OpenRouter LMM interaction

This notebook follows the pattern of the Perseus MCP examples, especially the OpenRouter tool-calling loop, but uses a local **Context-Fabric** MCP server against the N1904-TF corpus. The language model does not talk to Context-Fabric directly; this notebook is the bridge that:

1. starts `cfabric-mcp` for a selected corpus,
2. reads the MCP tool schemas,
3. sends those schemas to an OpenRouter chat model,
4. executes requested MCP tools locally, and
5. returns tool results to the model until it produces a final answer.

> Requirements: install `context-fabric[mcp]`, `mcp`, `httpx`, `python-dotenv`, and `ipython`; have network access to OpenRouter; and provide an OpenRouter API key. Clear notebook outputs before committing or sharing after a credentialed run.


## Configuration

Create a `.env` file in this repository root or set environment variables before launching Jupyter:

```dotenv
OPENROUTER_API_KEY=sk-or-v1-...
# Optional: choose another OpenRouter model with tool-calling support
OPENROUTER_MODEL=nvidia/nemotron-3-super-120b-a12b:free
# Optional: point to a local or downloadable Context-Fabric/Text-Fabric corpus
CFABRIC_CORPUS=centerblc/N1904/tf/1.0.0
# Optional: override the executable if needed
CFABRIC_MCP_COMMAND=cfabric-mcp
```

OpenRouter documentation: [quickstart](https://openrouter.ai/docs/quickstart) and [tool calling](https://openrouter.ai/docs/guides/features/tool-calling). Context-Fabric documents the MCP server as `cfabric-mcp --corpus /path/to/corpus`.


In [1]:
from contextlib import asynccontextmanager
from getpass import getpass
from pathlib import Path
import json
import os
import shlex
import sys

import httpx
from dotenv import load_dotenv
from IPython.display import Markdown, display

# MCP Python SDK imports.
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

REPO_ROOT = Path.cwd()
CFABRIC_LOG_DIR = REPO_ROOT / ".logs"
CFABRIC_ERRLOG = CFABRIC_LOG_DIR / "cfabric-mcp.stderr.log"
load_dotenv(REPO_ROOT / ".env", override=False)

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODEL = os.getenv(
    "OPENROUTER_MODEL", "nvidia/nemotron-3-super-120b-a12b:free"
)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("OpenRouter API key: ")

CFABRIC_CORPUS = os.getenv("CFABRIC_CORPUS", "centerblc/N1904/tf/1.0.0")
CFABRIC_CORPUS_NAME = os.getenv("CFABRIC_CORPUS_NAME", "N1904")
CFABRIC_MCP_WRAPPER = REPO_ROOT / "cfabric_mcp_wrapper.py"
## Default to the local wrapper so the spawned MCP process inherits the Windows path fix.
CFABRIC_MCP_COMMAND = os.getenv("CFABRIC_MCP_COMMAND") or f'"{sys.executable}" "{CFABRIC_MCP_WRAPPER}"'


def resolve_cfabric_corpus_path(corpus_value):
    """Resolve the corpus location to a real local path for cfabric-mcp."""
    raw_path = Path(corpus_value).expanduser()
    candidates = []

    if raw_path.is_absolute():
        candidates.append(raw_path)
    else:
        candidates.append((REPO_ROOT / raw_path).resolve())
        candidates.append((REPO_ROOT.parent / raw_path).resolve())

    # Try the sibling workspaces that are commonly used during local development.
    if corpus_value == "centerblc/N1904/tf/1.0.0":
        candidates.append((REPO_ROOT.parent / "cf_test" / "centerblc" / "N1904" / "tf" / "1.0.0").resolve())
        candidates.append((REPO_ROOT.parent / "FORK-CenterBLC-N1904" / "tf" / "1.0.0").resolve())

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        candidate_str = str(candidate)
        if candidate_str not in seen:
            seen.add(candidate_str)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate

    checked = "\n".join(f"- {candidate}" for candidate in unique_candidates)
    raise FileNotFoundError(
        f"Could not resolve CFABRIC_CORPUS={corpus_value!r}. Checked:\n{checked}"
    )


CFABRIC_CORPUS_PATH = resolve_cfabric_corpus_path(CFABRIC_CORPUS)


def make_cfabric_location_arg(corpus_path):
    """Return a cfabric-compatible corpus path argument."""
    # cfabric on Windows is happier with repo-relative paths than with raw drive-letter paths.
    try:
        relative_path = corpus_path.relative_to(REPO_ROOT)
    except ValueError:
        relative_path = Path(os.path.relpath(corpus_path, REPO_ROOT))
    return relative_path.as_posix()


CFABRIC_CORPUS_ARG = make_cfabric_location_arg(CFABRIC_CORPUS_PATH)

if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is required.")

print(f"OpenRouter model: {OPENROUTER_MODEL}")
print(f"Context-Fabric corpus setting: {CFABRIC_CORPUS}")
print(f"Corpus name: {CFABRIC_CORPUS_NAME}")
print(f"Resolved corpus path: {CFABRIC_CORPUS_PATH}")
print(f"MCP corpus argument: {CFABRIC_CORPUS_ARG}")
print(f"MCP command: {CFABRIC_MCP_COMMAND}")


OpenRouter model: nvidia/nemotron-3-super-120b-a12b:free
Context-Fabric corpus setting: centerblc/N1904/tf/1.0.0
Corpus name: N1904
Resolved corpus path: D:\Onedrive\GitHub\cf_test\centerblc\N1904\tf\1.0.0
MCP corpus argument: ../cf_test/centerblc/N1904/tf/1.0.0
MCP command: "c:\Users\tonyj\anaconda3\envs\context_fabric\python.exe" "d:\Onedrive\GitHub\CF_test_N1904-TF\cfabric_mcp_wrapper.py"


## Start a Context-Fabric MCP session

The helper below creates a stdio MCP client session. Keep all MCP calls inside the `async with cfabric_session()` block so the server process is cleaned up automatically.


In [2]:
def build_cfabric_server_params():
    """Build stdio server parameters for cfabric-mcp."""
    command_parts = shlex.split(CFABRIC_MCP_COMMAND)
    command = command_parts[0]
    # Load the corpus under a stable alias so downstream tool calls can address it by name.
    args = [*command_parts[1:], "--corpus", f"{CFABRIC_CORPUS_NAME}={CFABRIC_CORPUS_ARG}"]
    return StdioServerParameters(command=command, args=args, cwd=str(REPO_ROOT))


@asynccontextmanager
async def cfabric_session():
    """Return an initialized MCP session context manager for Context-Fabric."""
    CFABRIC_LOG_DIR.mkdir(exist_ok=True)
    with CFABRIC_ERRLOG.open("a", encoding="utf-8") as errlog:
        # Jupyter's stderr stream has no fileno(); swapping in a real file handle avoids that MCP startup bug.
        original_stderr = sys.stderr
        sys.stderr = errlog
        try:
            async with stdio_client(build_cfabric_server_params(), errlog=errlog) as streams:
                yield streams
        finally:
            sys.stderr = original_stderr


## Convert MCP tools to OpenRouter function tools

OpenRouter accepts OpenAI-compatible tool schemas. MCP tools already expose a name, description, and JSON input schema, so the conversion is mechanical.


In [3]:
# Probe the MCP server once so the rest of the notebook can reuse the discovered tool schemas.
async with cfabric_session() as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        listed_tools = await session.list_tools()

mcp_tools = listed_tools.tools
# OpenRouter expects OpenAI-style function specs, so reshape the MCP tool metadata here.
openrouter_tools = [
    {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.inputSchema or {"type": "object", "properties": {}},
        },
    }
    for tool in mcp_tools
]

print(f"Exposing {len(openrouter_tools)} Context-Fabric MCP tools to the LMM:")
print(", ".join(tool["function"]["name"] for tool in openrouter_tools))


Exposing 10 Context-Fabric MCP tools to the LMM:
list_corpora, describe_corpus, describe_feature, list_features, get_text_formats, search, search_continue, search_syntax_guide, get_passages, get_node_features


## LMM and MCP tool loop

The loop below sends the conversation and tool schemas to OpenRouter, executes any requested Context-Fabric MCP tools, appends the tool results, and repeats until the selected model returns a final answer.


In [4]:
def content_to_text(content):
    """Extract displayable text from MCP content blocks."""
    pieces = []
    for block in content or []:
        text = getattr(block, "text", None)
        if text is not None:
            pieces.append(text)
        else:
            pieces.append(json.dumps(block.model_dump(mode="json"), ensure_ascii=False))
    return "\n".join(pieces)


def openrouter_completion(messages):
    # Keep the HTTP call small and synchronous; the tool loop around it is already async.
    response = httpx.post(
        OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://github.com/CenterBLC/N1904",
            "X-Title": "Context-Fabric N1904 MCP notebook",
        },
        json={
            "model": OPENROUTER_MODEL,
            "messages": messages,
            "tools": openrouter_tools,
            "tool_choice": "auto",
        },
        timeout=120.0,
    )
    try:
        response.raise_for_status()
    except httpx.HTTPStatusError as exc:
        raise RuntimeError(
            f"OpenRouter error {response.status_code}: {response.text[:2000]}"
        ) from exc
    return response.json()


async def run_lmm_with_cfabric(user_prompt, max_rounds=8, max_tool_result_chars=20_000):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful Greek New Testament corpus research assistant. "
                "Use the Context-Fabric MCP tools for factual corpus lookup, "
                "structural search, feature inspection, and passage retrieval. "
                "Prefer discovery tools before calling tools with specific node ids, "
                "and cite the corpus references or node identifiers returned by tools."
            ),
        },
        {"role": "user", "content": user_prompt},
    ]

    async with cfabric_session() as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            for round_number in range(1, max_rounds + 1):
                # Each round lets the model either answer directly or request another MCP tool call.
                completion = openrouter_completion(messages)
                message = completion["choices"][0]["message"]
                assistant_message = {"role": "assistant", "content": message.get("content")}
                if message.get("tool_calls"):
                    assistant_message["tool_calls"] = message["tool_calls"]
                messages.append(assistant_message)

                display(Markdown(f"### LMM round {round_number}"))
                if message.get("content"):
                    display(Markdown(message["content"]))

                tool_calls = message.get("tool_calls") or []
                if not tool_calls:
                    return message.get("content", ""), messages

                for tool_call in tool_calls:
                    tool_name = tool_call["function"]["name"]
                    # MCP arguments arrive as a JSON string inside the OpenRouter tool-call envelope.
                    tool_args = json.loads(tool_call["function"].get("arguments") or "{}")
                    display(Markdown(
                        f"**Tool request:** `{tool_name}`\n\n"
                        f"```json\n{json.dumps(tool_args, ensure_ascii=False, indent=2)}\n```"
                    ))

                    try:
                        result = await session.call_tool(tool_name, tool_args)
                        result_text = content_to_text(result.content)
                    except Exception as exc:
                        result_text = f"Tool error: {type(exc).__name__}: {exc}"

                    # Trim oversized tool outputs so the next LLM request stays within a manageable context window.
                    tool_content = result_text[:max_tool_result_chars]
                    if len(result_text) > max_tool_result_chars:
                        tool_content += "\n...[tool result truncated]"

                    display(Markdown(
                        f"**Tool result:** `{tool_name}`\n\n"
                        f"```text\n{tool_content[:4000]}\n```"
                    ))
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call["id"],
                            "content": tool_content,
                        }
                    )

    raise RuntimeError(f"The model did not produce a final answer within {max_rounds} rounds.")


## Run an example N1904 research request

This prompt is intentionally corpus-grounded: the model should use Context-Fabric tools to discover the available structure and retrieve evidence rather than relying on memory.


In [5]:
prompt = (
    "Use the Context-Fabric MCP tools for the N1904 corpus to inspect John 1:1. "
    "Retrieve the Greek text, list the word-level lemmas and glosses if those features are available, "
    "and briefly explain which corpus references or node identifiers you used."
)

final_answer, conversation = await run_lmm_with_cfabric(prompt)
display(Markdown("## Final answer"))
display(Markdown(final_answer))


### LMM round 1

**Tool request:** `list_corpora`

```json
{}
```

**Tool result:** `list_corpora`

```text
{
  "corpora": [
    "N1904"
  ],
  "current": "N1904"
}
```

### LMM round 2

**Tool request:** `describe_corpus`

```json
{
  "corpus": "N1904"
}
```

...

**Tool result:** `get_node_features`

```text
{
  "nodes": [
    {
      "node": 49033,
      "type": "word",
      "lemma": "ἐν",
      "gloss": "(with dat.) in"
    },
    {
      "node": 49034,
      "type": "word",
      "lemma": "ἀρχή",
      "gloss": "beginning; ruler, authority"
    },
    {
      "node": 49035,
      "type": "word",
      "lemma": "εἰμί",
      "gloss": "am, exist"
    },
    {
      "node": 49036,
      "type": "word",
      "lemma": "ὁ",
      "gloss": "the"
    },
    {
      "node": 49037,
      "type": "word",
      "lemma": "λόγος",
      "gloss": "word, speech"
    },
    {
      "node": 49038,
      "type": "word",
      "lemma": "καί",
      "gloss": "and, also, likewise"
    },
    {
      "node": 49039,
      "type": "word",
      "lemma": "ὁ",
      "gloss": "the"
    },
    {
      "node": 49040,
      "type": "word",
      "lemma": "λόγος",
      "gloss": "word, speech"
    },
    {
      "node": 49041,
      "type": "word",
      "lemma": "εἰμί",
      "gloss": "am, exist"
    },
    {
      "node": 49042,
      "type": "word",
      "lemma": "πρός",
      "gloss": "(with acc.) to, towards, with, (with dat.) near, at"
    },
    {
      "node": 49043,
      "type": "word",
      "lemma": "ὁ",
      "gloss": "the"
    },
    {
      "node": 49044,
      "type": "word",
      "lemma": "θεός",
      "gloss": "god"
    },
    {
      "node": 49045,
      "type": "word",
      "lemma": "καί",
      "gloss": "and, also, likewise"
    },
    {
      "node": 49046,
      "type": "word",
      "lemma": "θεός",
      "gloss": "god"
    },
    {
      "node": 49047,
      "type": "word",
      "lemma": "εἰμί",
      "gloss": "am, exist"
    },
    {
      "node": 49048,
      "type": "word",
      "lemma": "ὁ",
      "gloss": "the"
    },
    {
      "node": 49049,
      "type": "word",
      "lemma": "λόγος",
      "gloss": "word, speech"
    }
  ],
  "features_requested": [
    "lemma",
    "gloss"
  ],
  "total": 17
}
```

### LMM round 8

**John 1:1 (N1904 corpus)**  
Greek text (as stored in the verse node):  
`Ἐν ἀρχῇ ἦν ὁ Λόγος, καὶ ὁ Λόγος ἦν πρὸς τὸν Θεόν, καὶ Θεὸς ἦν ὁ Λόγος.`  

**Word‑by‑word lemmas and glosses**  

| Word (surface) | Node ID | Lemma | Gloss (English) |
|----------------|--------|-------|-----------------|
| Ἐν | 49033 | ἐν | (with dat.) in |
| ἀρχῇ | 49034 | ἀρχή | beginning; ruler, authority |
| ἦν | 49035 | εἰμί | am, exist |
| ὁ | 49036 | ὁ | the |
| Λόγος, | 49037 | λόγος | word, speech |
| καὶ | 49038 | καί | and, also, likewise |
| ὁ | 49039 | ὁ | the |
| Λόγος | 49040 | λόγος | word, speech |
| ἦν | 49041 | εἰμί | am, exist |
| πρὸς | 49042 | πρός | (with acc.) to, towards, with, (with dat.) near, at |
| τὸν | 49043 | ὁ | the |
| Θεόν, | 49044 | θεός | god |
| καὶ | 49045 | καί | and, also, likewise |
| Θεὸς | 49046 | θεός | god |
| ἦν | 49047 | εἰμί | am, exist |
| ὁ | 49048 | ὁ | the |
| Λόγος. | 49049 | λόγος | word, speech |

**How the data were obtained**

1. **Verse retrieval** – I first looked up the verse reference `John 1:1` using `get_passages`, which returned the verse node **385605** (otype = `verse`) and its Greek text.  
2. **Word identification** – To isolate the words belonging to that verse, I used a search template `word book=John chapter=1 verse=1`. This yielded the 17 consecutive word nodes **49033–49049** (otype = `word`), each with its surface text and section reference confirming they are from John 1:1.  
3. **Lemma & gloss extraction** – Finally, I fetched the `lemma` and `gloss` features for those word nodes via `get_node_features`, producing the lemma and gloss pairs shown above.

Thus, the Greek text, lemmas, and glosses for John 1:1 are drawn directly from the N1904 corpus using the verse node 385605 and the word nodes 49033‑49049.

## Final answer

**John 1:1 (N1904 corpus)**  
Greek text (as stored in the verse node):  
`Ἐν ἀρχῇ ἦν ὁ Λόγος, καὶ ὁ Λόγος ἦν πρὸς τὸν Θεόν, καὶ Θεὸς ἦν ὁ Λόγος.`  

**Word‑by‑word lemmas and glosses**  

| Word (surface) | Node ID | Lemma | Gloss (English) |
|----------------|--------|-------|-----------------|
| Ἐν | 49033 | ἐν | (with dat.) in |
| ἀρχῇ | 49034 | ἀρχή | beginning; ruler, authority |
| ἦν | 49035 | εἰμί | am, exist |
| ὁ | 49036 | ὁ | the |
| Λόγος, | 49037 | λόγος | word, speech |
| καὶ | 49038 | καί | and, also, likewise |
| ὁ | 49039 | ὁ | the |
| Λόγος | 49040 | λόγος | word, speech |
| ἦν | 49041 | εἰμί | am, exist |
| πρὸς | 49042 | πρός | (with acc.) to, towards, with, (with dat.) near, at |
| τὸν | 49043 | ὁ | the |
| Θεόν, | 49044 | θεός | god |
| καὶ | 49045 | καί | and, also, likewise |
| Θεὸς | 49046 | θεός | god |
| ἦν | 49047 | εἰμί | am, exist |
| ὁ | 49048 | ὁ | the |
| Λόγος. | 49049 | λόγος | word, speech |

**How the data were obtained**

1. **Verse retrieval** – I first looked up the verse reference `John 1:1` using `get_passages`, which returned the verse node **385605** (otype = `verse`) and its Greek text.  
2. **Word identification** – To isolate the words belonging to that verse, I used a search template `word book=John chapter=1 verse=1`. This yielded the 17 consecutive word nodes **49033–49049** (otype = `word`), each with its surface text and section reference confirming they are from John 1:1.  
3. **Lemma & gloss extraction** – Finally, I fetched the `lemma` and `gloss` features for those word nodes via `get_node_features`, producing the lemma and gloss pairs shown above.

Thus, the Greek text, lemmas, and glosses for John 1:1 are drawn directly from the N1904 corpus using the verse node 385605 and the word nodes 49033‑49049.

## Inspect the complete message history

The conversation includes the original prompt, assistant tool requests, tool results, and final answer. Avoid committing outputs from credentialed runs.


In [6]:
print(json.dumps(conversation, ensure_ascii=False, indent=2))


[
  {
    "role": "system",
    "content": "You are a careful Greek New Testament corpus research assistant. Use the Context-Fabric MCP tools for factual corpus lookup, structural search, feature inspection, and passage retrieval. Prefer discovery tools before calling tools with specific node ids, and cite the corpus references or node identifiers returned by tools."
  },
  {
    "role": "user",
    "content": "Use the Context-Fabric MCP tools for the N1904 corpus to inspect John 1:1. Retrieve the Greek text, list the word-level lemmas and glosses if those features are available, and briefly explain which corpus references or node identifiers you used."
  },
  {
    "role": "assistant",
    "content": null,
    "tool_calls": [
      {
        "type": "function",
        "index": 0,
        "id": "call_c16486521f5b46a6a79bedda",
        "function": {
          "name": "list_corpora",
          "arguments": "{}"
        }
      }
    ]
  },
  {
    "role": "tool",
    "tool_call_id": "ca